# Actividad 5 - Clasificación con dataset Titanic

Este notebook prepara los datos, compara dos modelos de clasificación, ajusta hiperparámetros y registra resultados en MLflow.


In [ ]:
import os
from pathlib import Path

import mlflow
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path.cwd().resolve().parent
RAW_PATH = ROOT / 'datos' / 'datos_ini' / 'titanic.csv'
CLEAN_PATH = ROOT / 'datos' / 'datos_limp' / 'titanic_limpio.csv'
MLFLOW_TRACKING_URI = f"file://{ROOT / 'mlruns'}"
os.makedirs(ROOT / 'mlruns', exist_ok=True)
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment('Titanic_Classification')


In [ ]:
def clean_dataset(df):
    cleaned = df.copy()
    cleaned = cleaned.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], errors='ignore')
    for col in ['Age', 'Fare']:
        if col in cleaned.columns:
            cleaned[col] = cleaned[col].fillna(cleaned[col].median())
    for col in ['Embarked', 'Sex']:
        if col in cleaned.columns:
            cleaned[col] = cleaned[col].fillna(cleaned[col].mode()[0])
    if 'Survived' in cleaned.columns:
        cleaned['Survived'] = cleaned['Survived'].astype(int)
    cleaned['Sex'] = cleaned['Sex'].astype(str).str.lower()
    cleaned['Embarked'] = cleaned['Embarked'].astype(str).str.strip().str.upper()
    return cleaned

raw_df = pd.read_csv(RAW_PATH)
clean_df = clean_dataset(raw_df)
clean_df.to_csv(CLEAN_PATH, index=False)
clean_df.head()


In [ ]:
target = 'Survived'
X = clean_df.drop(columns=[target])
y = clean_df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = [col for col in X.columns if col not in numeric_features]

numeric_transformer = Pipeline([('imputer', SimpleImputer(strategy='median'))])
categorical_transformer = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features), ('cat', categorical_transformer, categorical_features)], remainder='drop')

models = {
    'logistic_regression': {
        'estimator': Pipeline([('preprocessor', preprocessor), ('model', LogisticRegression(max_iter=1000, random_state=42))]),
        'param_grid': {'model__C': [0.1, 1, 10], 'model__solver': ['liblinear', 'lbfgs']}
    },
    'random_forest': {
        'estimator': Pipeline([('preprocessor', preprocessor), ('model', RandomForestClassifier(random_state=42, n_estimators=200))]),
        'param_grid': {'model__n_estimators': [100, 200], 'model__max_depth': [None, 5, 10], 'model__min_samples_split': [2, 5]}
    }
}


In [ ]:
results = []
for name, config in models.items():
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    search = GridSearchCV(config['estimator'], param_grid=config['param_grid'], cv=cv, scoring='f1', n_jobs=-1)
    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1] if hasattr(best_model, 'predict_proba') else None
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
    }
    if y_proba is not None:
        metrics['roc_auc'] = roc_auc_score(y_test, y_proba)
    with mlflow.start_run(run_name=name) as run:
        mlflow.log_params({str(k): str(v) for k, v in search.best_params_.items()})
        for k, v in metrics.items():
            mlflow.log_metric(k, float(v))
        mlflow.sklearn.log_model(best_model, artifact_path='model')
        mlflow.log_text(classification_report(y_test, y_pred), f'{name}_classification_report.txt')
    results.append({'model': name, **metrics, 'best_params': search.best_params_})

pd.DataFrame(results)


## Conclusión

La comparación entre los modelos permite seleccionar la mejor configuración según la métrica principal (F1 o ROC AUC), y MLflow conserva la trazabilidad de parámetros, métricas y artefactos del entrenamiento.
